# Trust Score Demo

This notebook demonstrates how to implement and evaluate **data trust scores** using Kest. 

We will set up an OPA policy that enforces a minimum trust score threshold on data, establish data with varying trust labels originating from the internet, and show how trust scores can be manipulated during the lineage pipeline.

In [ ]:
from kest import config, originate, verified
from kest.core.policy import LocalOpaEngine, _HAS_REGORUS

if not _HAS_REGORUS:
    print("Error: lakera-regorus is not installed. Please install 'kest[opa]' on Python 3.11.")

config.policy_engine = LocalOpaEngine()
policy = """
package kest.trust
default allow = false

allow {
    input.trust_score >= 0.70
}
"""
config.policy_engine.add_policy("trust_access", policy)
print("Global OPA Engine instantiated with threshold >= 0.70")

### Defining Verification Pipelines

By default, `kest_verified` functions will inherently compute the **minimum** trust score of their parents when traversing lineages. 
However, we can explicitly define custom `trust_score_updaters` for specific nodes.

In [ ]:
@verified(trust_score_updater=lambda scores: max(scores) + 0.3 if scores else 0.8)
def validate_and_clean(data: dict) -> dict:
    print("-> Cleaning and validating data. Trust score upgrading...")
    cleaned = data.copy()
    cleaned["validated"] = True
    return cleaned

@verified(enforce_rules=["data.kest.trust.allow"])
def generate_report(data: dict) -> dict:
    print(f"-> Successfully generated report from high-trust data: {data}")
    return {"report_generated": True, "data": data}

### Scenario 1: Processing Without Validation (Blocked)

If we take raw untrustworthy data and route it straight into a secured endpoint, it should fail.

In [ ]:
# Our data entered the system with a modest trust score of 0.5
raw_data = originate({"source": "website", "content": "raw_user_data"}, trust_score=0.5)

try:
    generate_report(raw_data)
except PermissionError as e:
    print(f"\n❌ Blocked Successfully! Reason: {e}")

### Scenario 2: With Validation (Allowed)

If we run the data through our `validate_and_clean` pipeline, the trust score is dynamically augmented across the DAG, enabling it to hit the necessary reporting threshold.

In [ ]:
clean_data = validate_and_clean(raw_data)

# Output will be allowed, and history tracked
report = generate_report(clean_data)

leaf_id = list(report.passport.history.keys())[-1]
final_score = report.passport.history[leaf_id].trust_score

print(f"\n✅ Report Generated! Final Trusted DAG Node Score: {final_score}")